In [7]:
from seismic import SeismicIndexDotVByte, SeismicIndex

In [12]:
json_input_file = ""
index = SeismicIndexDotVByte.build(json_input_file)


Building the index (standard u16)...
Configuration { pruning: GlobalThreshold { n_postings: 3500, max_fraction: 1.5 }, blocking: RandomKmeans { centroid_fraction: 0.1, min_cluster_size: 2, clustering_algorithm: RandomKmeansInvertedIndexApprox { doc_cut: 15 } }, summarization: EnergyPreserving { summary_energy: 0.4 }, knn: KnnConfiguration { nknn: 0, knn_path: None } }
Reading the collection..
Number of rows: 522931
Elapsed time to read the collection: 14 secs
Distributing and pruning postings: 8 secs
Number of posting lists: 26595
Avg posting list length: 512.07
Building summaries: 2 secs
Converting to DotVByte...


In [16]:
#index = SeismicIndex.build(json_input_file)


Building the index...
Configuration { pruning: GlobalThreshold { n_postings: 3500, max_fraction: 1.5 }, blocking: RandomKmeans { centroid_fraction: 0.1, min_cluster_size: 2, clustering_algorithm: RandomKmeansInvertedIndexApprox { doc_cut: 15 } }, summarization: EnergyPreserving { summary_energy: 0.4 }, knn: KnnConfiguration { nknn: 0, knn_path: None } }
Reading the collection..
Number of rows: 522931
Elapsed time to read the collection: 14 secs
Distributing and pruning postings: 8 secs
Number of posting lists: 26595
Avg posting list length: 512.07
Building summaries: 2 secs


In [17]:
import numpy as np
import json

file_path = ""

queries = []
with open(file_path, 'r') as f:
    for line in f:
        queries.append(json.loads(line))

MAX_TOKEN_LEN = 30
string_type  = f'U{MAX_TOKEN_LEN}'

queries_ids = np.array([q['id'] for q in queries], dtype=string_type)

query_components = []
query_values = []

for query in queries:
    vector = query['vector']
    query_components.append(np.array(list(vector.keys()), dtype=string_type))
    query_values.append(np.array(list(vector.values()), dtype=np.float32))
    
results = index.batch_search(
    queries_ids=queries_ids,
    query_components=query_components,
    query_values=query_values,
    k=10,
    query_cut=10,
    heap_factor=0.7,
    n_knn=0,
    sorted=True, 
    num_threads=1,
)

In [18]:
import ir_measures
import ir_datasets

# add your ir_dataset dataset string id below, e.g., "beir/quora/test"
ir_dataset_string = "beir/quora/test"

ir_results = [ir_measures.ScoredDoc(query_id, doc_id, score) for r in results for (query_id, score, doc_id) in r]
qrels = ir_datasets.load(ir_dataset_string).qrels

In [19]:
metric_name = "NDCG@10" # on BEIR/quora
ir_measure = ir_measures.parse_measure(metric_name)
ir_measures.calc_aggregate([ir_measure], qrels, ir_results)

{nDCG@10: 0.8141847914619893}